# Aufgaben Blatt 4 KI Machine Learning I



## Aufgabe 1 (Lineare Regression)

Bearbeiten Sie die Aufgabe https://github.com/oduerr/ki/blob/main/linear_regression/lr_gradient_descent.ipynb

Versuchen Sie den Code zu verstehen und machen die kleineren Aufgaben, die in dem notebook besprochen werden.

## Aufgabe 2 (Titanic)
In dieser Aufgabe nehmen Sie an der Titanic Challenge (https://www.kaggle.com/c/titanic) teil. Sie können die Aufgabe am eigenen PC lösen oder direkt in Kaggle lösen. Die Daten liegen auch auf Moodle. 

a) Lesen Sie die Trainingsdaten ein und teilen Sie sie in ein Validierungsdatenset (20%) und in ein eigentliches Trainigsdatenset (80%) auf. Finden Sie auf dem Trainigsdatenset eine Regel für das Überleben alleine aufgrund der Klasse des Tickets (Pclass). Wenden Sie diese Regel auf die Validierungsdaten an. Wie gut ist die Genauigkeit (Anteil der korrekten Klassifikationen) auf den Validierungsdaten?  

In [88]:
# Hinweise zum Einlesen
import pandas as pd
import sklearn
from sklearn.linear_model import LogisticRegression

all_values = pd.read_csv('titanic-20260530/train.csv')

all_values = pd.read_csv('titanic-20260530/train.csv')

train_val = all_values.sample(frac=0.8, random_state=42).copy()
test_values = all_values.drop(train_val.index).copy()

age_median = train_val["Age"].median(skipna=True)

train_val["Age"] = train_val["Age"].fillna(age_median)
test_values["Age"] = test_values["Age"].fillna(age_median)
print("train_val")
print(train_val)
print("test_values")
print(test_values)
# Hinweise zum Erzeugen einer Tabelle
#pd.crosstab(...)
num_people = train_val.shape[0]
print(num_people)
classes = train_val['Pclass']

class_prob = [0,0,0]
for p_class in range(0,3):
    num_people_survived_of_current_class = 0
    total_count_in_class = 0
    for current_row, current_class in classes.items():
        if current_class != (p_class + 1):
            continue
        if train_val.loc[current_row]['Survived']:
            num_people_survived_of_current_class += 1
        total_count_in_class += 1
    class_prob[p_class] = num_people_survived_of_current_class / total_count_in_class


for current_class, prob in enumerate(class_prob, start=1):
    print(f"Probability of surviving of class {current_class}: {prob}")




# Hinweise um die Accuracy zu berechnen
from sklearn.metrics import accuracy_score

train_val
     PassengerId  Survived  Pclass  \
709          710         1       3   
439          440         0       2   
840          841         0       3   
720          721         1       2   
39            40         1       3   
..           ...       ...     ...   
639          640         0       3   
878          879         0       3   
824          825         0       3   
803          804         1       3   
619          620         0       2   

                                                  Name     Sex    Age  SibSp  \
709  Moubarek, Master. Halim Gonios ("William George")    male  28.00      1   
439             Kvillner, Mr. Johan Henrik Johannesson    male  31.00      0   
840                        Alhomaki, Mr. Ilmari Rudolf    male  20.00      0   
720                  Harper, Miss. Annie Jessie "Nina"  female   6.00      0   
39                         Nicola-Yarred, Miss. Jamila  female  14.00      1   
..                                                 ..

b) Wenden Sie die Regel aus a) auf die Testdaten an und laden Sie Ihre Lösung hoch. 

In [89]:
#############################################
# Hinweise zum rausschreiben
import random
def survices_prediction(p_class):
    return int(class_prob[p_class -1 ] >= 0.5)

#Enhält die Predictions 0 für Tod 1 für Survived
predictions = []
actual_values = []

for _, person in test_values.iterrows():
    current_class = person["Pclass"]
    actually_survived = person["Survived"]

    pred_survived = survices_prediction(current_class)

    predictions.append(pred_survived)
    actual_values.append(actually_survived)

print("Accuracy:", accuracy_score(actual_values, predictions))

Accuracy: 0.6685393258426966


c) Logistische Regression mit Pclass

Trainieren Sie eine logistische Regression mit den Variablen 'Pclass'. Verwenden Sie die Klasse `sklearn.linear_model.LogisticRegression`. Berechnen Sie die Accuracy auf dem Validierungsset.

In [90]:
from sklearn.linear_model import LogisticRegression
logReg = LogisticRegression()
x = train_val[['Pclass']]
y = train_val['Survived']
logReg.fit(x, y)


X_test = test_values[['Pclass']]
predictions = logReg.predict(X_test)
y_test = test_values['Survived'].astype(int)
accuary_linReg = accuracy_score(y_test, predictions)
print(f"Accuracy", accuary_linReg)


Accuracy 0.6685393258426966


d) Coding / Feature engineering 

d.i) Missing Values:

Verwenden Sie nun weitere Features. Die Variable Age enthält Missing values, die Sie durch folgenden code ersetzen können (was passiert da?)

In [91]:
test_values["Age"].fillna(train_val["Age"].median(skipna=True), inplace=True)
train_val["Age"].fillna(train_val["Age"].median(skipna=True), inplace=True)

C:\Users\tangu\AppData\Local\Temp\ipykernel_34180\727244271.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  test_values["Age"].fillna(train_val["Age"].median(skipna=True), inplace=True)
C:\Users\tangu\AppData\Local\Temp\ipykernel_34180\727244271.py:2: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series

709    28.00
439    31.00
840    20.00
720     6.00
39     14.00
       ...  
639    28.00
878    28.00
824     2.00
803     0.42
619    26.00
Name: Age, Length: 713, dtype: float64

d.ii) Kategorische Variable

Verwenden Sie die Funktion `pd.get_dummies` um die Variablen 'Pclass' and 'Sex' in numerische Werte umzuwandeln. Führen Sie nun eine logistische Regression durch.

e) Weitere Klassifikatoren. Neben der logistischen Regression, gibt es weitere Klassifikatoren. Der Random-Forest ist ein recht stabiler Klassifikator, was wäre die Performance von diesem Klassifikator.

In [85]:
# Hinweise zur Lösung
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier()
#rf hat nun gleiches interface, wie die logistische Regression

f) [optional] Versuchen Sie weitere Features zu erzeugen und laden den besten Klassifikator auf Kaggle hoch. 

## Aufgabe 3 Titanic mit Neuronalen Netzen 

Hinweis: Diese Aufgabe kann erst nach der dritten Vorlesung in ML gemacht werden.

Mit den gleichen Daten, wie in der Aufgabe 2 d. Erstellen Sie ein fully connected neural network und fitten es an die Ttrainingsdaten. Verwenden Sie mindestens zwei hidden Layer. Plotten Sie den Verlauf der Loss Kurve für die Trainings- und Validierungsdaten. Optional: Laden Sie Ihre beste Lösung auf Kaggle hoch. 

In [86]:
#Sie können von folgendem Code starten um das Netzwerk zu definieren, füllen Sie die ...
import tensorflow as tf
import tensorflow.keras 
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
model = Sequential()
model.add(Dense(..., activation='sigmoid', batch_input_shape=(None, 4))) #We have 4 input features
#...
model.add(Dense(1, activation='sigmoid'))
opt = tf.keras.optimizers.Adam(learning_rate=1e-3)
model.compile(loss=tf.keras.losses.BinaryCrossentropy(),
              optimizer=opt,
              metrics=['accuracy'])
model.summary()

ModuleNotFoundError: No module named 'tensorflow'